In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
from penaltyblog.models import DixonColesGoalModel
from penaltyblog.ratings import Elo
import seaborn as sns
import matplotlib.pyplot as plt
from penaltyblog.models import create_dixon_coles_grid

In [3]:
df = pd.read_csv('../jogos-selecoes.csv')
df['date'] = pd.to_datetime(df['date'])
df.head()

,date,home_team,away_team,home_score,away_score,tournament,country,neutral
0,1960-01-01,Morocco,Yugoslavia,0,5,Friendly,Morocco,False
1,1960-01-03,Tunisia,Yugoslavia,1,5,Friendly,Tunisia,False
2,1960-01-06,Italy,Switzerland,3,0,Central European International Cup,Italy,False
3,1960-01-08,Egypt,Yugoslavia,0,1,Friendly,United Arab Republic,True
4,1960-01-27,Guinea-Bissau,Gambia,3,2,Friendly,Portuguese Guinea,False


In [4]:
# ajuste de peso: quanto mais antiga menos peso tem. Partidas até 3 anos atrás (2023) tem peso 1 e a cada 3 anos cai pela metade
# ou seja, entre 4 e 6 anos pesam 0.5, entre 7 e 9 pesam 0.25, entre 10 e 12 anos pesam 0.125 até o peso mínimo de 0.015. O que vier após isso (21 anos) será deletado
current_year = datetime.now().year

# remove os antigos
least_date = datetime(current_year - 21, 1, 1)
df = df[df.date >= least_date]

# adiciona coluna de peso
def define_peso(date):
    current_year = datetime.now().year
    diferenca = current_year - date.year
    expoente = diferenca // 3
    return (1/2.0)**(expoente)

df['peso-tempo'] = df.date.apply(define_peso)
df.head()

,date,home_team,away_team,home_score,away_score,tournament,country,neutral,peso-tempo
23939,2005-01-02,Singapore,Myanmar,4,2,AFF Championship,Singapore,False,0.007812
23940,2005-01-03,Malaysia,Indonesia,1,4,AFF Championship,Malaysia,False,0.007812
23941,2005-01-08,Egypt,Uganda,3,0,Friendly,Egypt,False,0.007812
23942,2005-01-08,Indonesia,Singapore,1,3,AFF Championship,Indonesia,False,0.007812
23943,2005-01-08,Jamaica,French Guiana,5,0,CFU Caribbean Cup qualification,Jamaica,False,0.007812


In [5]:
# segundo ajuste de peso. O tipo de campeonato interfere no quanto os jogadores dão garra. Portanto seguimos a tabela abaixo para pontuar de acordo com a competição
# Copa do mundo: 1
# Continental: 0.9
# elimatorias: 0.8
# eliminatorias pro continental: 0.6
# amistosos: 0.5
# demais: 0.5
def define_peso_competicao(tournament):
    copa_mundo = 'FIFA World Cup'
    continentais = ['UEFA Euro', 'African Cup of Nations', 'Copa América', 'Gold Cup', 'AFC Asian Cup', 'Oceania Nations Cup']
    eliminatorias_copa = 'FIFA World Cup qualification'
    eliminatorias_continental = ['AFC Asian Cup qualification', 'UEFA Euro qualification', 'African Cup of Nations qualification', 'Gold Cup qualification', 
                                 'Copa América qualification', 'CONCACAF Nations League qualification', 'Gold Cup qualification']
    if(tournament == copa_mundo): return 1.0
    if(tournament in continentais): return 0.9
    if(tournament == eliminatorias_copa): return 0.8
    if(tournament in eliminatorias_continental): return 0.6
    return 0.5

df['peso-competicao'] = df.tournament.apply(define_peso_competicao)

df.head()

,date,home_team,away_team,home_score,away_score,tournament,country,neutral,peso-tempo,peso-competicao
23939,2005-01-02,Singapore,Myanmar,4,2,AFF Championship,Singapore,False,0.007812,0.5
23940,2005-01-03,Malaysia,Indonesia,1,4,AFF Championship,Malaysia,False,0.007812,0.5
23941,2005-01-08,Egypt,Uganda,3,0,Friendly,Egypt,False,0.007812,0.5
23942,2005-01-08,Indonesia,Singapore,1,3,AFF Championship,Indonesia,False,0.007812,0.5
23943,2005-01-08,Jamaica,French Guiana,5,0,CFU Caribbean Cup qualification,Jamaica,False,0.007812,0.5


In [6]:
# junta todos os pesos em um só
df['peso'] = df['peso-tempo'] * df['peso-competicao']
df = df.drop(columns=['peso-tempo', 'peso-competicao'])

df.head()

,date,home_team,away_team,home_score,away_score,tournament,country,neutral,peso
23939,2005-01-02,Singapore,Myanmar,4,2,AFF Championship,Singapore,False,0.003906
23940,2005-01-03,Malaysia,Indonesia,1,4,AFF Championship,Malaysia,False,0.003906
23941,2005-01-08,Egypt,Uganda,3,0,Friendly,Egypt,False,0.003906
23942,2005-01-08,Indonesia,Singapore,1,3,AFF Championship,Indonesia,False,0.003906
23943,2005-01-08,Jamaica,French Guiana,5,0,CFU Caribbean Cup qualification,Jamaica,False,0.003906


In [8]:
# treina o modelo usando o peso com vários fatores que inventamos
modelo = DixonColesGoalModel(
  df["home_score"].to_numpy(copy=True),
  df["away_score"].to_numpy(copy=True),
  df["home_team"].to_numpy(copy=True), # to_numpy porque a biblioteca do modelo trabalha com números, não com dataframe e o copy é para que ele possa fazer alterações internas nos dados sem afetar os originais
  df["away_team"].to_numpy(copy=True),
  weights=df["peso"].to_numpy(copy=True),
) # params: gols do time da casa, gols do visitante, nome do time da casa, nome do visitante. Peso é opcional

modelo.fit() # define um valor de ataque e de defesa para cada time, uma constante de vantagem para o dono da casa
params = modelo.get_params()
rho = params['rho']
vantagem_mandante = params['home_advantage']

In [19]:
# calculando K como a média dos campeonatos na base de dados
# segundo a biblioteca, um site importante aplica Elo diferentes para cada torneio (60 copa, 50 continental, 40 eliminatorias, 20 amistoso e 30 os demais)
# aplicaremos esse valores pros torneios e tirar a média para ter o Elo final
def valor_k_competicao(tournament):
    copa_mundo = 'FIFA World Cup'
    continentais = ['UEFA Euro', 'African Cup of Nations', 'Copa América', 'Gold Cup', 'AFC Asian Cup', 'Oceania Nations Cup']
    eliminatorias_copa = 'FIFA World Cup qualification'
    eliminatorias_continental = ['AFC Asian Cup qualification', 'UEFA Euro qualification', 'African Cup of Nations qualification', 'Gold Cup qualification', 
                                 'Copa América qualification', 'CONCACAF Nations League qualification', 'Gold Cup qualification']
    if(tournament == copa_mundo): return 60
    if(tournament in continentais): return 50
    if(tournament == eliminatorias_copa): return 40
    if(tournament in eliminatorias_continental): return 40
    if(tournament == 'Friendly'): return 20
    return 30

k_values = df.tournament.apply(valor_k_competicao)
k = sum(k_values)/len(k_values)

elo = Elo(k=k, home_field_advantage=75) # por default todo mundo começa com 1500 e vai mudando a cada jogo do dataset

print(f'Valor de K: {k}')

Valor de K: 32.28699111132937


In [22]:
# atualiza o ELO apos cada jogo
for i in range(len(df)):
    line = df.iloc[i]
    mandante_nome = line['home_team']
    visitante_nome = line['away_team']
    mandante_gols = line['home_score']
    visitante_gols = line['away_score']
    elo_casa = elo.get_team_rating(mandante_nome)
    elo_visit = elo.get_team_rating(visitante_nome)

    if(mandante_gols > visitante_gols):
        result = 0
    else:
        if(mandante_gols == visitante_gols):
            result = 1 
        else: 
            result = 2
    elo.update_ratings(mandante_nome, visitante_nome, result)

print(f'Elo Mexico: {elo.get_team_rating('Mexico')}. Coreia do Sul: {elo.get_team_rating('South Korea')}')

Elo Mexico: 1805.0832034110551. Coreia do Sul: 1794.1275181645867


In [23]:
# calcula os gols esperados do primeiro jogo da copa
atk_mexico = params['attack_Mexico']
def_mexico = params['defence_Mexico']
atk_coreia = params['attack_South Korea']
def_coreia = params['defence_South Korea']

expect_gols_mex = np.exp( atk_mexico + def_coreia)
expect_gols_cor = np.exp( atk_coreia + def_mexico)
elo_mex = elo.get_team_rating('Mexico')
elo_cor = elo.get_team_rating('South Korea')

# Peso jogo: 80% dixon-coles e 20% ELO
peso_partida = 0.8 * (expect_gols_mex - expect_gols_cor) + 0.2 * (elo_mex - elo_cor)
print(f'Peso da partida: {peso_partida}')

soma_lambdas = expect_gols_mex + expect_gols_cor
# atualiza os lambdas
expect_gols_mex = (soma_lambdas + peso_partida)/2 # mandante recebe a media das lambdas com o peso
expect_gols_cor = (soma_lambdas - peso_partida)/2 # visitante recebe a diferença média das lambdas com o peso

# como lambda representa os gols esperados, ñ pode ser negativo. Portanto vamos considerar o menor valor como 0.05 (proximo de 0, mas com alguma chance de fazer gol ainda)
expect_gols_mex = max(expect_gols_mex, 0.05)
expect_gols_cor = max(expect_gols_cor, 0.05)

# soma expectativa adicional de gols para o mandante
# no caso o mandante é o vasco
expect_gols_mex *= np.exp(vantagem_mandante)

print(f'Expectativa de gols mexico: {expect_gols_mex:.4f}. Expectativa de gols Coreia do Sul: {expect_gols_cor:.4f}')

Peso da partida: 2.3088702365844087
Expectativa de gols mexico: 2.6782. Expectativa de gols Coreia do Sul: 0.0500


# RODANDO SÓ 1 VEZ - apenas pega o resultado com maior chance de acontecer

In [25]:
# retorna o mesmo objeto do predict, portanto esse método faz a mesma função do predict 
# para quando vc já tem a previsão de gols vindas de algum lugar (como um bolão ou bet) ou quando quer dar seus proprios pesos a chance de gols
previsao = create_dixon_coles_grid(expect_gols_mex, expect_gols_cor, rho, max_goals=7)
matriz_placar = previsao.grid

print(f'Chance do mandante ganhar: {previsao.home_win}')
print(f'Chance de empate: {previsao.draw}')
print(f'Chance do visitante ganhar: {previsao.away_win}')

max_val = matriz_placar.max()
gols_mandante, gols_visitante = np.unravel_index(np.argmax(matriz_placar), matriz_placar.shape)

print(f"Placar mais provável: Mexico {gols_mandante} x Coreia do Sul {gols_visitante}")

Chance do mandante ganhar: 0.9188635182971122
Chance de empate: 0.07746221537430303
Chance do visitante ganhar: 0.0036742663285847335
Placar mais provável: Mexico 2 x Coreia do Sul 0


# FAZENDO 50_000 SIMULAÇÕES DE MONTE CARLO - calcula a chance de todos os placares 1x só e depois sorteia dessa lista 50mil vezes de acordo com a probabilidade de cada um

In [51]:
# transforma a matriz em um array com as probabilidades
probabilidades = np.array(matriz_placar).reshape(-1)

# Normalizar as probabilidades para garantir que somem exatamente 1.0 (exigência do numpy). Antes estava dando 0.9999999999
probabilidades /= probabilidades.sum()

sum(probabilidades)

np.float64(1.0)

In [68]:
num_simulacoes = 50_000

# Sorteia os índices (placares) com base na probabilidade (É AQUI QUE A SIMULAÇÃO É EXECUTADA)
indices_sorteados = np.random.choice(len(placares), size=num_simulacoes, p=probabilidades)

# Contadores dos resultados simulados
vitorias_mexico = 0
empates = 0
vitorias_coreia = 0

for idx in indices_sorteados:
    gols_mexico = idx//8 # recupera qual era a linha da matriz (mexico era o mandante) (pega a divisao inteira por 8 pq o tamanho de cad alinha é 8, de 0 a 7)
    gols_coreia = idx % 8 # recupera qual a coluna da matriz (coreia era o visitante) (o resto da divisão por 8 dá a coluna)
    if(gols_mexico > gols_coreia): vitorias_mexico+=1
    elif(gols_mexico < gols_coreia): vitorias_coreia+=1
    else: empates+=1

indice_mais_repetido = np.argmax(np.bincount(indices_sorteados))
jogo_mais_rep_quantidade = (indices_sorteados == indice_mais_repetido).sum()
jogo_mais_rep_gols_mexico = indice_mais_repetido//8
jogo_mais_rep_gols_coreia = indice_mais_repetido % 8

print(f'México venceu {vitorias_mexico} jogos ({(100*vitorias_mexico/num_simulacoes):.2f})%')
print(f'Coreia venceu {vitorias_coreia} jogos ({(100*vitorias_coreia/num_simulacoes):.2f})%')
print(f'Empate em {empates} jogos ({(100*empates/num_simulacoes):.2f})%')

jogo_mais_rep_quantidade
print(f'Placar mais repetido: Mexico {jogo_mais_rep_gols_mexico} x Coreia {jogo_mais_rep_gols_coreia} se repetiu {(100 * jogo_mais_rep_quantidade/num_simulacoes):.2f}% das vezes')

México venceu 45912 jogos (91.82)%
Coreia venceu 169 jogos (0.34)%
Empate em 3919 jogos (7.84)%
Placar mais repetido: Mexico 2 x Coreia 0 se repetiu 24.06% das vezes
